Impor Data

In [ ]:
import os
import pandas as pd

# Folder proyek = induk dari folder code/. Dapat diganti lewat FPL_BASE_DIR.
BASE_DIR = os.environ.get("FPL_BASE_DIR", os.path.dirname(os.getcwd()))
DATA_DIR = os.path.join(BASE_DIR, "data")

PERIOD_FOLDERS = [
    "GW 13 (Data GW 1-12)", "GW 16 (Data GW 1-15)",
    "GW 19 (Data GW 1-18)", "GW 22 (Data GW 1-21)",
    "GW 26 (Data GW 1-25)", "GW 30 (Data GW 1-29)",
    "GW 33 (Data GW 1-32)", "GW 36 (Data GW 1-35)",
]

def load_raw(folder_name):
    return pd.read_csv(os.path.join(DATA_DIR, folder_name, "raw", "players_raw_20252026.csv"))

# Periode pertama dipakai untuk memeriksa struktur kolom
df_2526 = load_raw(PERIOD_FOLDERS[0])

In [ ]:
print("Informasi kolom musim 2025/2026:")
df_2526.info()

In [ ]:
print(df_2526.head())

In [ ]:
desired_columns = [
    'first_name',
    'second_name',
    'team',
    "element_type",
    'minutes',
    'goals_scored',
    'assists',
    'clean_sheets',
    'goals_conceded',
    'own_goals',
    'penalties_saved',
    'penalties_missed',
    'yellow_cards',
    'red_cards',
    'saves',
    'starts',
    'bonus',
    'bps',
    'total_points',
    'now_cost',
    'influence',
    'creativity',
    'threat',
    'ict_index',
    'expected_goals',
    'expected_assists',
    'expected_goal_involvements',
    'expected_goals_conceded'
]

common_columns = list(df_2526.columns)
final_columns = [col for col in desired_columns if col in common_columns]
missing_columns = [col for col in desired_columns if col not in common_columns]

print("Kolom yang dipilih:")
print(final_columns)

In [ ]:
def preprocess_single_season(df, season, columns):
    df = df[columns].copy()
    df['season'] = season
    
    position_mapping = {1: 'GK', 2: 'DEF', 3: 'MID', 4: 'FWD'}
    df['position'] = df['element_type'].map(position_mapping)
    
    df = df.rename(columns={'goals_scored': 'goals', 'now_cost': 'price'})
    df = df.drop(columns=['element_type'])
    df['price'] = df['price'] / 10
    
    df = df.drop_duplicates()
    df = df.dropna(subset=['position'])
    df = df[df['minutes'] > 0].copy()
    
    df['goals_per_90'] = (df['goals'] / df['minutes']) * 90
    df['assists_per_90'] = (df['assists'] / df['minutes']) * 90
    df['clean_sheets_per_90'] = (df['clean_sheets'] / df['minutes']) * 90
    df['points_per_90'] = (df['total_points'] / df['minutes']) * 90
    df['influence_per_90'] = (df['influence'] / df['minutes']) * 90
    df['creativity_per_90'] = (df['creativity'] / df['minutes']) * 90
    df['threat_per_90'] = (df['threat'] / df['minutes']) * 90
    df['ict_index_per_90'] = (df['ict_index'] / df['minutes']) * 90
    df['points_per_90'] = (df['total_points'] / df['minutes']) * 90
    
    first_columns = ['first_name', 'second_name', 'position', 'season', 'minutes', 'price']
    other_columns = [col for col in df.columns if col not in first_columns]
    df = df[first_columns + other_columns]
    
    return df

In [ ]:
for folder_name in PERIOD_FOLDERS:
    df_raw = load_raw(folder_name)
    absent = [col for col in desired_columns if col not in df_raw.columns]
    if absent:
        raise ValueError(f"{folder_name}: kolom tidak ada di data mentah: {absent}")
    df_processed = preprocess_single_season(df_raw, "2025/2026", desired_columns)
    out_dir = os.path.join(DATA_DIR, folder_name, "processed", "musim 2025-2026")
    os.makedirs(out_dir, exist_ok=True)
    df_processed.to_csv(os.path.join(out_dir, "players_processed_20252026.csv"), index=False)
    print(f"{folder_name}: {len(df_processed)} pemain, {df_processed.shape[1]} kolom")